# Identificação de Variáveis Preditoras (feature selection)

## Teoria
* É o processo de escolhas das variáveis com o maior nível de significancia para o modelo.
* Utilizamos essa técnica para evitar um fenômeno chamado Maldição de dimensionalidade e evitar o over-fitting.

Um bom artigo para entender melhor o processo:

https://medium.com/datarisk-io/feature-selection-24338ce39067

## Pacotes

In [0]:
%run
Utils/configuration

imblearn is installed

Out[3]: DataFrame[]

In [0]:
from IPython.display import Image
from IPython.core.display import HTML

## Análise prévia de preparação

* Para utilização das funções abaixo, se faz necessário executar uma análise prévia de correlação e colinearidade. Assim, o modelo em questão fará uma escolha refinada das melhores variáveis.
* Esse processo estará explicado no caderno de exemplo no link a seguir:

# Métodos

## Visão Geral

* O objetivo de ambas as funções é devolver um Dataset com as melhores preditoras numéricas para os modelos que serão trabalhados.
* São englobadas nesse processo apenas variáveis do tipo **numéricas**, porém, a função identifica variáveis categóricas e as exclui da análise em questão
* Cada função tem o objetivo de extrair as colunas das variáveis preditoras, passá-las por um modelo de arvore de decisão, executar uma análise de significancia dessas variáveis **setado pelo usuário após sua própria análise**, filtrar as melhores preditoras de acordo com um ranking de porcentagem, que é criado pelo modelo e por fim retornar o Dataframe já pronto para uso nos próximos passos.
* A **diferença** entre as duas **funções** está relacionada ao **tipo de arvore de decisão** que será utlizado. São elas **Random Forest Classifier** ou **Extra Trees Classifier**.
* A explicação dos modelos estão explicados em suas respectivas comands.

##selecao_RFC

### Randon Forest Classifier
---
https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html

* A ideia principal do RFC e do ETC reside no fato de que, em vez de calcular a divisão localmente ideal para uma combinação de recursos, um valor aleatório é selecionado para a divisão da árvore extra. Assim, a aleatoriedade vai um passo adiante na forma como as divisões são calculadas. Então, a ideia é, em vez de não gastar tempo descobrindo o melhor ponto de divisão, eu pego aleatoriamente um ponto e divido com base no que isso leva a árvores mais diversificadas e menos divisores para obscurecer ao treinar uma floresta extremamente aleatória.
* Ambos os modelos performam na mesma proporção em termos de acuracia, curva ROC ou F1 score

* Arvores de Decisão no gera: É um modelo não paramétrico **supervisionado** para Classificação e regressão
* Esse modelo utilizado a seguir é uma vertente do modelo tradicional de arvore de decisão tradicional de classificação. Onde cada árvore é construída a partir de uma amostra bootstrap do conjunto de treinamento. Utlizamos os método **feature_importances_** para extrair o nível de significância e a porcentagem de explicabilidade das variáveis preditoras em função da variável target.
* Ao dividir cada nó na construção da árvore, a melhor divisão é encontrada por meio dos recursos de entrada ou o analista pode setar o numero máximo de features pelo hiperparâmetro **max_features**
* O objetivo dessa fonte de aleatoriedade é diminuir a variância do estimador florestal. De fato, as árvores de decisão individuais normalmente exibem alta variância e tendem a se ajustar demais. A aleatoriedade injetada nas florestas produz árvores de decisão com erros de previsão um tanto desacoplados. Ao fazer uma média dessas previsões, alguns erros podem ser cancelados. Florestas aleatórias atingem uma variação reduzida combinando diversas árvores, às vezes ao custo de um leve aumento no viés. Na prática, a redução da variância é muitas vezes significativa, resultando em um modelo globalmente melhor.

### Exemplos Gráficos

In [0]:
Image(url = 'https://miro.medium.com/max/1200/1*hmtbIgxoflflJqMJ_UHwXw.jpeg')

Out[47]:

### Função

In [0]:
def selecao_RFC(X,y,p, n_estimators=100, criterion='gini', max_depth=None, min_samples_split=2, min_samples_leaf=1, min_weight_fraction_leaf=0.0, max_features='auto', max_leaf_nodes=None, min_impurity_decrease=0.0, bootstrap=True, oob_score=False, n_jobs=None, random_state=None, verbose=0, warm_start=False, class_weight=None, ccp_alpha=0.0, max_samples=None):
  # https://adb-6226397775262915.15.azuredatabricks.net/?o=6226397775262915#notebook/1121831065220356/command/3749511783433129
  '''
  X = Dataframe com as variáveis preditoras do tipo numérica em dataframe Pandas
  y = Dataframe com as variáveis targets do tipo numérica
  p = valor int ou float do nível de significancia que deseja obter dos dados
  retorno = um dataset com as melhores variáveis preditoras para o modelo em questão
  '''
  # separação de variáveis numéricas e categóricas e lista das colunas
  categoricas = X.select_dtypes('category')
  variaveis_categoricas = categoricas.columns
  numericas = X.drop(categoricas, axis = 1)
  variaveis = numericas.columns
  
  # teste de verificação entre variáveis numéricas e categóricas
  if len(variaveis_categoricas) != 0:
    print('As variáveis categoricas {} serão extraídas para o modelo de análise'.format(list(categoricas.columns)))
  else:
    print('Só existem variáveis numéricas.')
  
  # análise de variáveis categóricas  com o modelo Random forest
  clf = RandomForestClassifier(n_estimators=n_estimators, criterion=criterion, max_depth=max_depth, min_samples_split=min_samples_split, min_samples_leaf=min_samples_leaf, min_weight_fraction_leaf=min_weight_fraction_leaf, max_features=max_features, max_leaf_nodes=max_leaf_nodes, min_impurity_decrease=min_impurity_decrease, bootstrap=bootstrap, oob_score=oob_score, n_jobs=n_jobs, random_state=random_state, verbose=verbose, warm_start=warm_start, class_weight=class_weight, ccp_alpha=ccp_alpha, max_samples=max_samples)
  clf.fit(numericas,y)
  features = clf.feature_importances_
  features = features.astype('float')
  
  # seletor de variáveis e criação de coluna da porcentagem de significância
  valores = np.array([variaveis, features])
  resultado = pd.DataFrame(valores)
  resultado = resultado.T
  resultado = resultado.rename(columns = {0: 'variavel', 1: 'valores'})
  resultado = resultado.sort_values(by = 'valores', ascending = False)
  resultado['valores'] = resultado['valores'].astype('float')
  resultado['porcentos'] = [i * 100 for i in resultado['valores']]
  p = p
  var = 0
  feature = []
  for i in resultado['porcentos']:
    var = var + i
    feature.append(var)
  resultado['soma'] = feature
  
  # filtra as melhores variáveis
  subset = resultado[resultado['soma'] < p]
  
  # filtrar melhores variáveis numéricas e devolver as variáveis categóricas para o dataset
  numerics = X[subset['variavel'].values]
  numerics['key'] = [i + 1 for i in  range(0, len(numerics))]
  categoricas['key'] = [i + 1 for i in  range(0, len(categoricas))]
  dataframe = numerics.set_index('key').join(categoricas.set_index('key'), on = 'key')
  
  #leitura de componentes para impressão das informações
  linhas = len(dataframe.index)
  colunas = len(numerics.columns)
  columns = subset['variavel'].values
  print('As variáveis numéricas possuem {}% de significância'.format(p))
  print('O número de linhas é de {} e de melhores preditoras numéricas de {}'.format(linhas, colunas))
  print('O modelo selecionou as variáveis: {}'.format(columns))
  print('O Número de árvores: {}'.format(n_estimators))
  print('A função da qualidade da divisão entre "gini", "log-loss" e "entropia" é: {}'.format(criterion))
  print('A profundidade máxima da árvore: {}'.format(max_depth))
  print('O número mínimo de amostras necessárias para dividir um nó: {}'.format(min_samples_split))
  print('O número mínimo de amostras necessárias para estar em um nó folha: {}'.format(min_samples_leaf))
  print('A fração ponderada mínima da soma total de pesos: {}'.format(min_weight_fraction_leaf))
  print('O número de recursos a serem considerados ao procurar a melhor divisão: {}'.format(max_features))
  print('O número maximo dos melhores nós: {}'.format(max_leaf_nodes))
  print('O valor do decrescimo das impurezas: {}'.format(min_impurity_decrease))
  print('A Suavização bootstrap: {}'.format(bootstrap))
  print('O out of bag score: {}'.format(oob_score))
  print('O número de jobs paralelos: {}'.format(n_jobs))
  print('A definição da seed: {}'.format(random_state))
  print('A verbosidade do modelo: {}'.format(verbose))
  print('Reutilizador de ajustes da camada anterior: {}'.format(warm_start))
  print('Pesos associados às classes: {}'.format(class_weight))
  print('Parâmetro de complexidade: {}'.format(ccp_alpha))
  print('Se bootstrap for True, número máximo de amostras para X: {}'.format(max_samples))
  dataframe = spark.createDataFrame(dataframe)
  for i in variaveis_categoricas[0:len(variaveis_categoricas)]:
    dataframe = dataframe.withColumn(i, col(i).cast('string'))
  return dataframe

##selecao_etc

### Extra Tree Classifier
---
https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.ExtraTreesClassifier.html?highlight=extra%20tree%20classifier#sklearn.ensemble.ExtraTreesClassifier

* O modelo selecionará amostras para cada arvore de decisão sem substituí-las, assim, todas as amostras serão unicas por natureza. 
* O número total das variáveis selecionadas ainda permanecerão as mesmas que a raiz quadrada do total do numero de variáveis se você estiver usando uma tarefa de classificação.
* Em vez de procurar os limites mais discriminativos, os limites são desenhados aleatoriamente para cada característica candidata e o melhor desses limites gerados aleatoriamente é escolhido como a regra de divisão.
* Isso ajuda a reduzir um pouco mais a variância do modelo, às custas de um aumento ligeiramente maior no viés.
* Da mesma forma que a função anterior este modelo utiliza o método **feature_importances_** para extrair o nível de significância e a porcentagem de explicabilidade das variáveis preditoras em função da variável target.
* Aparentemente são mais rápidas, comparadas ao RFC

### Função

In [0]:
def selecao_etc(X,y,p, n_estimators=100, criterion='gini', max_depth=None, min_samples_split=2, min_samples_leaf=1, min_weight_fraction_leaf=0.0, max_features='auto', max_leaf_nodes=None, min_impurity_decrease=0.0, bootstrap=False, oob_score=False, n_jobs=None, random_state=None, verbose=0, warm_start=False, class_weight=None, ccp_alpha=0.0, max_samples=None):
  # https://adb-6226397775262915.15.azuredatabricks.net/?o=6226397775262915#notebook/1121831065220356/command/3749511783433128
  '''
  X = Dataframe com as variáveis preditoras do tipo numérica em dataframe Pandas
  y = Dataframe com as variáveis targets do tipo numérica
  p = valor int ou float do nível de significância que deseja obter dos dados
  retorno = um dataset com as melhores variáveis preditoras para o modelo em questão
  '''
  # separação de variáveis numéricas e categóricas e lista das colunas
  categoricas = X.select_dtypes('category')
  variaveis_categoricas = categoricas.columns
  numericas = X.drop(categoricas, axis = 1)
  variaveis = numericas.columns
  if len(variaveis_categoricas) != 0:
    print('As variáveis categoricas {} serão extraídas para o modelo de análise'.format(list(categoricas.columns)))
  else:
    print('Só existem variáveis numéricas.')
  
  # análise de variáveis categóricas  com o modelo Random forest
  clf = ExtraTreesClassifier(n_estimators=n_estimators, criterion=criterion, max_depth=max_depth, min_samples_split=min_samples_split, min_samples_leaf=min_samples_leaf, min_weight_fraction_leaf=min_weight_fraction_leaf, max_features=max_features, max_leaf_nodes=max_leaf_nodes, min_impurity_decrease=min_impurity_decrease, bootstrap=bootstrap, oob_score=oob_score, n_jobs=n_jobs, random_state=random_state, verbose=verbose, warm_start=warm_start, class_weight=class_weight, ccp_alpha=ccp_alpha, max_samples=max_samples)
  clf.fit(numericas,y)
  features = clf.feature_importances_
  features = features.astype('float')
  
  # seletor de variáveis e criação de coluna da porcentagem de significância
  valores = np.array([variaveis, features])
  resultado = pd.DataFrame(valores)
  resultado = resultado.T
  resultado = resultado.rename(columns = {0: 'variavel', 1: 'valores'})
  resultado = resultado.sort_values(by = 'valores', ascending = False)
  resultado['valores'] = resultado['valores'].astype('float')
  resultado['porcentos'] = [i * 100 for i in resultado['valores']]
  p = p
  var = 0
  feature = []
  for i in resultado['porcentos']:
    var = var + i
    feature.append(var)
  resultado['soma'] = feature
  
  # filtra as melhores variáveis
  subset = resultado[resultado['soma'] < p]
  
  # seleciona as melhores variáveis numéricas e devolver as variáveis categóricas para o dataset
  numerics = X[subset['variavel'].values]
  numerics['key'] = [i + 1 for i in  range(0, len(numerics))]
  categoricas['key'] = [i + 1 for i in  range(0, len(categoricas))]
  dataframe = numerics.set_index('key').join(categoricas.set_index('key'), on = 'key')
  
  
  # leitura de componentes para impressão das informações
  linhas = len(dataframe.index)
  colunas = len(numerics.columns)
  columns = subset['variavel'].values
  print('As variáveis numéricas possuem {}% de significância'.format(p))
  print('O número de linhas é de {} e de melhores preditoras numéricas {}'.format(linhas, colunas))
  print('O modelo selecionou as variáveis: {}'.format(columns))
  print('O Número de árvores: {}'.format(n_estimators))
  print('A função da qualidade da divisão entre "gini", "log-loss" e "entropia" é: {}'.format(criterion))
  print('A profundidade máxima da árvore: {}'.format(max_depth))
  print('O número mínimo de amostras necessárias para dividir um nó: {}'.format(min_samples_split))
  print('O número mínimo de amostras necessárias para estar em um nó folha: {}'.format(min_samples_leaf))
  print('A fração ponderada mínima da soma total de pesos: {}'.format(min_weight_fraction_leaf))
  print('O número de recursos a serem considerados ao procurar a melhor divisão: {}'.format(max_features))
  print('O número máximo dos melhores nós: {}'.format(max_leaf_nodes))
  print('O valor do decrescimo das impurezas: {}'.format(min_impurity_decrease))
  print('A Suavização bootstrap: {}'.format(bootstrap))
  print('O out of bag score: {}'.format(oob_score))
  print('O número de jobs paralelos: {}'.format(n_jobs))
  print('A definição da seed: {}'.format(random_state))
  print('A verbosidade do modelo: {}'.format(verbose))
  print('Reutilizador de ajustes da camada anterior: {}'.format(warm_start))
  print('Pesos associados a classes: {}'.format(class_weight))
  print('Parâmetro de complexidade: {}'.format(ccp_alpha))
  print('Se bootstrap for True, numero máximo de amostras para X: {}'.format(max_samples))
  dataframe = spark.createDataFrame(dataframe)
  for i in variaveis_categoricas[0:len(variaveis_categoricas)]:
    dataframe = dataframe.withColumn(i, col(i).cast('string'))
  return dataframe